In [2]:
import os
import sys
os.chdir("..")

import pandas as pd
import duckdb
from pathlib import Path
from config import RESEARCH_DB_PATH, NSE_DB_PATH

In [3]:
con = duckdb.connect(RESEARCH_DB_PATH)
con.execute(f"ATTACH '{NSE_DB_PATH}' AS nse (READ_ONLY)") 

# read from nse, write to research.db
con.execute("""
    CREATE OR REPLACE TABLE forward_returns AS
    WITH daily AS (
        SELECT
            trade_date,
            index_name,
            close
        FROM nse.market_activity_index
        WHERE index_name = 'Nifty 50'   -- adjust to exact name in your data
        ORDER BY trade_date
    )
    SELECT
        d.trade_date,
        d.index_name,
        d.close,

        -- forward returns
        ROUND((f1.close - d.close) / d.close * 100, 4) AS fwd_ret_1d,
        ROUND((f5.close - d.close) / d.close * 100, 4) AS fwd_ret_5d,
        ROUND((f20.close - d.close) / d.close * 100, 4) AS fwd_ret_20d,

        -- direction label (for classification)
        CASE WHEN f1.close > d.close THEN 1 ELSE 0 END AS up_1d

    FROM daily d
    LEFT JOIN daily f1
        ON f1.trade_date = (
            SELECT MIN(trade_date) FROM daily
            WHERE trade_date > d.trade_date
        )
    LEFT JOIN daily f5
        ON f5.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 4
        )
    LEFT JOIN daily f20
        ON f20.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 19
        )
    ;
""")

In [4]:
results = con.execute(""" SELECT * FROM forward_returns ORDER BY trade_date DESC LIMIT 10 OFFSET 20; """).fetchall()
for row in results:
    print(row)

(datetime.date(2026, 5, 13), 'Nifty 50', 23412.6, 1.1831, 1.0524, -1.0721, 1)
(datetime.date(2026, 5, 12), 'Nifty 50', 23379.55, 0.1414, 1.0199, -0.704, 1)
(datetime.date(2026, 5, 11), 'Nifty 50', 23815.85, -1.832, -0.6966, -2.4091, 0)
(datetime.date(2026, 5, 8), 'Nifty 50', 24176.15, -1.4903, -2.2032, -4.3562, 0)
(datetime.date(2026, 5, 7), 'Nifty 50', 24326.65, -0.6187, -2.6187, -3.9461, 0)
(datetime.date(2026, 5, 6), 'Nifty 50', 24330.95, -0.0177, -3.7744, -3.7582, 0)
(datetime.date(2026, 5, 5), 'Nifty 50', 24032.8, 1.2406, -2.7182, -2.6098, 1)
(datetime.date(2026, 5, 4), 'Nifty 50', 24119.3, -0.3586, -1.2581, -2.6359, 0)
(datetime.date(2026, 4, 30), 'Nifty 50', 23997.55, 0.5073, 0.7442, -2.5626, 1)
(datetime.date(2026, 4, 29), 'Nifty 50', 24177.65, -0.7449, 0.6163, -2.6053, 0)


In [5]:
results = con.execute(""" SELECT DISTINCT index_name FROM nse.market_activity_index ORDER BY 1; """).fetchall()
for row in results:
    print(row)

('BHARATBOND-APR25',)
('BHARATBOND-APR30',)
('BHARATBOND-APR31',)
('BHARATBOND-APR32',)
('BHARATBOND-APR33',)
('India VIX',)
('NIFTY Alpha 50',)
('NIFTY AlphaLowVol',)
('NIFTY CONSR DURBL',)
('NIFTY HEALTHCARE',)
('NIFTY IND DIGITAL',)
('NIFTY INDIA MFG',)
('NIFTY LARGEMID250',)
('NIFTY M150 QLTY50',)
('NIFTY MICROCAP250',)
('NIFTY MID SELECT',)
('NIFTY MIDCAP 100',)
('NIFTY MIDCAP 150',)
('NIFTY MIDSML 400',)
('NIFTY OIL AND GAS',)
('NIFTY SMLCAP 100',)
('NIFTY SMLCAP 250',)
('NIFTY SMLCAP 50',)
('NIFTY TOTAL MKT',)
('NIFTY100 EQL Wgt',)
('NIFTY100 ESG',)
('NIFTY100 LowVol30',)
('NIFTY100 Qualty30',)
('NIFTY200 QUALTY30',)
('NIFTY50 EQL Wgt',)
('NIFTY500 MULTICAP',)
('Nifty 100',)
('Nifty 200',)
('Nifty 50',)
('Nifty 500',)
('Nifty AQL 30',)
('Nifty AQLV 30',)
('Nifty Auto',)
('Nifty Bank',)
('Nifty CPSE',)
('Nifty Capital Mkt',)
('Nifty Cement',)
('Nifty Chemicals',)
('Nifty Commodities',)
('Nifty Consumption',)
('Nifty CoreHousing',)
('Nifty Corp MAATR',)
('Nifty Div Opps 50',)
('Ni

In [6]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close         AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close        AS vix_close

FROM forward_returns fr
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

ORDER BY fr.trade_date
""")

In [7]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10 OFFSET 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 5, 13), 23412.6, 1.1831, 1.0524, -1.0721, 1, 19.425)
(datetime.date(2026, 5, 12), 23379.55, 0.1414, 1.0199, -0.704, 1, 19.28)
(datetime.date(2026, 5, 11), 23815.85, -1.832, -0.6966, -2.4091, 0, 18.5525)
(datetime.date(2026, 5, 8), 24176.15, -1.4903, -2.2032, -4.3562, 0, 16.84)
(datetime.date(2026, 5, 7), 24326.65, -0.6187, -2.6187, -3.9461, 0, 16.62)
(datetime.date(2026, 5, 6), 24330.95, -0.0177, -3.7744, -3.7582, 0, 16.6775)
(datetime.date(2026, 5, 5), 24032.8, 1.2406, -2.7182, -2.6098, 1, 17.9075)
(datetime.date(2026, 5, 4), 24119.3, -0.3586, -1.2581, -2.6359, 0, 18.3)
(datetime.date(2026, 4, 30), 23997.55, 0.5073, 0.7442, -2.5626, 1, 18.46)
(datetime.date(2026, 4, 29), 24177.65, -0.7449, 0.6163, -2.6053, 0, 17.4375)


In [8]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 13 THEN '1_low <13'
            WHEN vix_close < 16 THEN '2_calm 13-16'
            WHEN vix_close < 20 THEN '3_normal 16-20'
            WHEN vix_close < 25 THEN '4_elevated 20-25'
            ELSE                     '5_fear >25'
        END AS vix_regime,

        COUNT(*)                        AS days,
        ROUND(AVG(fwd_ret_1d), 3)       AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)       AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)      AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)      AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL

    GROUP BY 1
    ORDER BY 1
""").fetchall()

for row in results: print(row)

('1_low <13', 166, -0.019, -0.165, -0.533, 50.6)
('2_calm 13-16', 183, -0.067, -0.145, -0.377, 45.9)
('3_normal 16-20', 85, -0.075, -0.153, -0.205, 49.4)
('4_elevated 20-25', 20, 0.635, 1.275, 3.694, 65.0)
('5_fear >25', 6, 0.524, 4.01, 6.276, 83.3)


In [9]:
results = con.execute("""
    SELECT *
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
    ORDER BY trade_date DESC, expiry ASC
    LIMIT 20
""").fetchall()
for row in results: print(row)

('IDO', 'NIFTY', datetime.date(2026, 6, 16), datetime.date(2026, 6, 11), 118823900.0, 129258675.0, 0.9192721494321368, 23161.600000000028, 23200.0)
('IDO', 'NIFTY', datetime.date(2026, 6, 23), datetime.date(2026, 6, 11), 12525630.0, 13443235.0, 0.931742248052645, 23161.600000000053, 23300.0)
('IDO', 'NIFTY', datetime.date(2026, 6, 30), datetime.date(2026, 6, 11), 69842280.0, 66482435.0, 1.050537333658131, 23161.599999999897, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 7), datetime.date(2026, 6, 11), 420745.0, 788645.0, 0.5335036676831781, 23161.60000000006, 23300.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 14), datetime.date(2026, 6, 11), 44915.0, 53820.0, 0.8345410628019324, 23161.600000000057, 23500.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 28), datetime.date(2026, 6, 11), 10593895.0, 9133020.0, 1.159955305036012, 23161.60000000002, 23800.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 25), datetime.date(2026, 6, 11), 2180815.0, 1881620.0, 1.159009257979826, 23161.600000000064, 24000

In [10]:
results = con.execute("""
    SELECT
        expiry,
        trade_date,
        pe_oi,
        ce_oi,
        pe_oi + ce_oi AS total_oi
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND trade_date = '2026-06-10'
    ORDER BY total_oi DESC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 16), datetime.date(2026, 6, 10), 93749955.0, 114460970.0, 208210925.0)
(datetime.date(2026, 6, 30), datetime.date(2026, 6, 10), 69249000.0, 66301370.0, 135550370.0)
(datetime.date(2026, 12, 29), datetime.date(2026, 6, 10), 11812480.0, 9371595.0, 21184075.0)
(datetime.date(2026, 6, 23), datetime.date(2026, 6, 10), 9232405.0, 10127780.0, 19360185.0)
(datetime.date(2026, 7, 28), datetime.date(2026, 6, 10), 9825335.0, 8697390.0, 18522725.0)
(datetime.date(2026, 9, 29), datetime.date(2026, 6, 10), 5249010.0, 4500755.0, 9749765.0)
(datetime.date(2026, 8, 25), datetime.date(2026, 6, 10), 2054260.0, 1737060.0, 3791320.0)
(datetime.date(2026, 7, 7), datetime.date(2026, 6, 10), 275860.0, 505245.0, 781105.0)
(datetime.date(2027, 12, 28), datetime.date(2026, 6, 10), 218430.0, 155535.0, 373965.0)
(datetime.date(2028, 12, 26), datetime.date(2026, 6, 10), 44980.0, 16115.0, 61095.0)
(datetime.date(2026, 7, 14), datetime.date(2026, 6, 10), 9425.0, 12155.0, 21580.0)
(datetime.dat

In [11]:
results = con.execute("""
    SELECT
        percentile_cont(0.25) WITHIN GROUP (ORDER BY total_oi) AS p25,
        percentile_cont(0.50) WITHIN GROUP (ORDER BY total_oi) AS p50,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY total_oi) AS p75,
        percentile_cont(0.90) WITHIN GROUP (ORDER BY total_oi) AS p90,
        percentile_cont(0.95) WITHIN GROUP (ORDER BY total_oi) AS p95,
        MIN(total_oi)  AS min_oi,
        MAX(total_oi)  AS max_oi,
        COUNT(*)       AS total_rows
    FROM (
        SELECT pe_oi + ce_oi AS total_oi
        FROM nse.options_analytics
        WHERE ticker = 'NIFTY'
          AND pe_oi IS NOT NULL
          AND ce_oi IS NOT NULL
    )
""").fetchall()
for row in results: print(row)

(2825.0, 322882.5, 11805262.5, 86405625.00000001, 210933344.99999997, 0.0, 424741950.0, 8298)


In [12]:
results = con.execute("""
    SELECT
        trade_date,
        COUNT(*) AS liquid_expiries
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
    ORDER BY trade_date DESC
    LIMIT 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 11), 5)
(datetime.date(2026, 6, 10), 5)
(datetime.date(2026, 6, 9), 5)
(datetime.date(2026, 6, 8), 5)
(datetime.date(2026, 6, 5), 5)
(datetime.date(2026, 6, 4), 5)
(datetime.date(2026, 6, 3), 5)
(datetime.date(2026, 6, 2), 5)
(datetime.date(2026, 6, 1), 5)
(datetime.date(2026, 5, 29), 4)
(datetime.date(2026, 5, 27), 4)
(datetime.date(2026, 5, 26), 5)
(datetime.date(2026, 5, 25), 4)
(datetime.date(2026, 5, 22), 4)
(datetime.date(2026, 5, 21), 4)
(datetime.date(2026, 5, 20), 4)
(datetime.date(2026, 5, 19), 4)
(datetime.date(2026, 5, 18), 4)
(datetime.date(2026, 5, 15), 4)
(datetime.date(2026, 5, 14), 4)


In [13]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close              AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close             AS vix_close,
    pcr_agg.pcr,
    mp_agg.max_pain_dist_pct

FROM forward_returns fr

LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

-- PCR: all expiries
LEFT JOIN (
    SELECT
        trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: liquid expiries only, OI-weighted
LEFT JOIN (
    SELECT
        trade_date,
        ROUND(
            SUM(
                ((underlying - max_pain) / NULLIF(max_pain, 0) * 100)
                * (pe_oi + ce_oi)
            ) / NULLIF(SUM(pe_oi + ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [14]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 11), 23161.6, None, None, None, 0, 15.6125, 0.9853, -1.6451)
(datetime.date(2026, 6, 10), 23214.95, -0.2298, None, None, 0, 15.6325, 0.9344, -1.8159)
(datetime.date(2026, 6, 9), 23242.1, -0.1168, None, None, 0, 15.575, 0.9304, -0.9622)
(datetime.date(2026, 6, 8), 23123.0, 0.5151, None, None, 1, 17.0275, 0.7764, -1.5009)
(datetime.date(2026, 6, 5), 23366.7, -1.0429, None, None, 0, 15.7875, 0.8273, -1.307)
(datetime.date(2026, 6, 4), 23416.55, -0.2129, -1.0888, None, 0, 15.885, 1.0032, -1.2287)
(datetime.date(2026, 6, 3), 23405.6, 0.0468, -0.8145, None, 1, 16.2775, 1.0164, -1.5506)
(datetime.date(2026, 6, 2), 23483.55, -0.3319, -1.0282, None, 0, 15.355, 1.0541, -0.7102)
(datetime.date(2026, 6, 1), 23382.6, 0.4317, -1.1102, None, 1, 16.5425, 0.6929, -1.5239)
(datetime.date(2026, 5, 29), 23547.75, -0.7013, -0.7689, None, 0, 16.185, 0.7445, -1.904)


In [15]:
results = con.execute("""
    SELECT
        CASE
            WHEN pcr < 0.7  THEN '1_very_low <0.7'
            WHEN pcr < 0.9  THEN '2_low 0.7-0.9'
            WHEN pcr < 1.1  THEN '3_neutral 0.9-1.1'
            WHEN pcr < 1.3  THEN '4_high 1.1-1.3'
            ELSE                 '5_very_high >1.3'
        END AS pcr_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND pcr IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("=== PCR ===")
for row in results: print(row)

results = con.execute("""
    SELECT
        CASE
            WHEN max_pain_dist_pct < -3   THEN '1_far_below <-3%'
            WHEN max_pain_dist_pct < -1.5 THEN '2_below -3 to -1.5%'
            WHEN max_pain_dist_pct < 0    THEN '3_slightly_below -1.5 to 0%'
            WHEN max_pain_dist_pct < 1.5  THEN '4_slightly_above 0 to 1.5%'
            ELSE                               '5_far_above >1.5%'
        END AS mp_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND max_pain_dist_pct IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("\n=== Max Pain Distance ===")
for row in results: print(row)

=== PCR ===
('1_very_low <0.7', 25, 0.025, 0.112, -1.021, 52.0)
('2_low 0.7-0.9', 175, -0.072, -0.051, 0.189, 42.9)
('3_neutral 0.9-1.1', 153, 0.004, -0.082, -0.132, 50.3)
('4_high 1.1-1.3', 89, 0.046, 0.112, -0.236, 56.2)
('5_very_high >1.3', 18, 0.064, -0.458, -1.541, 72.2)

=== Max Pain Distance ===
('1_far_below <-3%', 5, 0.292, 0.385, 5.101, 80.0)
('2_below -3 to -1.5%', 27, 0.029, 0.199, 1.719, 66.7)
('3_slightly_below -1.5 to 0%', 286, -0.006, 0.047, -0.229, 46.2)
('4_slightly_above 0 to 1.5%', 141, -0.037, -0.263, -0.399, 52.5)
('5_far_above >1.5%', 1, -1.39, 0.083, 0.716, 0.0)


In [16]:
# Futures - what tickers and how many rows
results = con.execute("""
    SELECT instrument_type, ticker, COUNT(*) as rows, 
           MIN(trade_date) as from_date, MAX(trade_date) as to_date
    FROM nse.futures_analytics
    GROUP BY instrument_type, ticker
    ORDER BY rows DESC
    LIMIT 10
""").fetchall()
print("=== Futures ===")
for row in results: print(row)

=== Futures ===
('STF', 'GLENMARK', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('STF', 'DABUR', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('STF', 'SHRIRAMFIN', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('STF', 'LUPIN', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('STF', 'BAJAJFINSV', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('STF', 'SBICARD', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('STF', 'BANKBARODA', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('STF', 'BEL', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('STF', 'ALKEM', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))
('IDF', 'NIFTYNXT50', 1383, datetime.date(2024, 8, 1), datetime.date(2026, 6, 11))


In [18]:
# Participant - what participant types and asset classes exist
results = con.execute("""
    SELECT participant_type, metric_type, asset_class, direction, option_side,
           COUNT(*) as rows
    FROM nse.participant_activity
    GROUP BY participant_type, metric_type, asset_class, direction, option_side
    ORDER BY participant_type, metric_type, asset_class
    LIMIT 30
""").fetchall()
print("\n=== Participant ===")
for row in results: print(row)


=== Participant ===
('Client', 'OI', 'INDEX', 'short', 'CE', 461)
('Client', 'OI', 'INDEX', 'short', 'NA', 461)
('Client', 'OI', 'INDEX', 'long', 'PE', 461)
('Client', 'OI', 'INDEX', 'long', 'CE', 461)
('Client', 'OI', 'INDEX', 'short', 'PE', 461)
('Client', 'OI', 'INDEX', 'long', 'NA', 461)
('Client', 'OI', 'STOCK', 'long', 'PE', 461)
('Client', 'OI', 'STOCK', 'long', 'CE', 461)
('Client', 'OI', 'STOCK', 'long', 'NA', 461)
('Client', 'OI', 'STOCK', 'short', 'CE', 461)
('Client', 'OI', 'STOCK', 'short', 'PE', 461)
('Client', 'OI', 'STOCK', 'short', 'NA', 461)
('Client', 'VOL', 'INDEX', 'short', 'PE', 461)
('Client', 'VOL', 'INDEX', 'short', 'NA', 461)
('Client', 'VOL', 'INDEX', 'long', 'NA', 461)
('Client', 'VOL', 'INDEX', 'long', 'PE', 461)
('Client', 'VOL', 'INDEX', 'short', 'CE', 461)
('Client', 'VOL', 'INDEX', 'long', 'CE', 461)
('Client', 'VOL', 'STOCK', 'long', 'CE', 461)
('Client', 'VOL', 'STOCK', 'short', 'CE', 461)
('Client', 'VOL', 'STOCK', 'long', 'NA', 461)
('Client', 'VOL

In [23]:
results = con.execute("""
    SELECT DISTINCT(participant_type) as ptypes
    FROM nse.participant_activity
""").fetchall()
print("\n=== Participant Classes ===")
for row in results: print(row)


=== Participant Classes ===
('DII',)
('FII',)
('Pro',)
('Client',)
